# Crystal Structure Visualizer — Google Colab

An interactive tool to **upload, edit, and visualize crystallographic structures**, then export a **multi-panel PDF**.

**Features:** upload CIF/POSCAR/XYZ/XSF/PDB · bond-length cutoff (keeps van-der-Waals interlayer gaps) · per-pair bond cutoffs · symmetry from file (spglib) or manual `x,y,z` ops · edit cell size & angles · per-element colours · rotate to any angle · legend, cell-dimension labels, and an orientation compass — each independently positionable (pick a corner, then fine-tune with Move X/Y) and independently toggleable per PDF panel · supercell (e.g. 3×3×1) · append cells · delete individual atoms · PDF with custom proportions and per-panel view placement (with a one-click 'use current view').

**How to use:** Run the two cells below in order. The second cell shows the full UI. Upload a file with the **Upload** button in the *File / Cell* tab.

In [ ]:
#@title Step 1 — install dependencies (run once)
!pip install ase spglib ipywidgets -q
print('Dependencies installed. Now run the next cell.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.9 MB/s eta 0:00:00
Dependencies installed. Now run the next cell.


In [ ]:
#@title Step 2 — launch the visualizer

"""
Crystal Structure Visualizer — Google Colab edition
====================================================
An ipywidgets-based GUI for visualizing and editing crystallographic
structures, designed to run inside a Google Colab (or Jupyter) notebook.

Feature parity with the desktop version:
  * Upload CIF / POSCAR / XYZ / XSF / PDB (anything ASE reads) and edit them
  * Bond cut-off (max bond length) + per-element-pair overrides, so van-der-
    Waals layered crystals keep their interlayer gap
  * Detect symmetry (spglib) or apply manual `x,y,z` operations
  * Edit cell parameters a, b, c, alpha, beta, gamma (or the raw matrix)
  * Per-element atom colours
  * Rotate to any (elevation, azimuth); presets for a/b/c-axis and iso
  * Toggle legend, unit-cell edges, cell-dimension labels, perspective, etc.
  * Build supercells (e.g. 3x3x1) and append fractional cells around the box
  * Delete individual atoms (including appended ones)
  * Export a multi-panel PDF with custom page size / proportions and an
    independent view (angle, zoom, caption, legend) per panel

USAGE IN COLAB
--------------
    # Cell 1 — install dependencies
    !pip install ase spglib ipywidgets -q

    # Cell 2 — load and launch
    #   (upload this file with the Files pane, or %%writefile it, then:)
    from crystal_visualizer_colab import launch
    app = launch()

The `launch()` call displays the whole UI inline. Everything updates live.
"""

import os
import io
import tempfile
from itertools import product

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.backends.backend_pdf import PdfPages

# ---- ASE ------------------------------------------------------------------
from ase import Atoms
from ase.io import read, write
from ase.data import covalent_radii, atomic_numbers, chemical_symbols
from ase.data.colors import jmol_colors

try:
    from ase.spacegroup import get_spacegroup
    HAS_SPACEGROUP = True
except Exception:
    HAS_SPACEGROUP = False

try:
    from mpl_toolkits.mplot3d import proj3d
    HAS_PROJ3D = True
except Exception:
    HAS_PROJ3D = False

# ---- ipywidgets / IPython -------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output


# ===========================================================================
# Helpers
# ===========================================================================
PAGE_SIZES = {
    "A0":      (33.11, 46.81),
    "A1":      (23.39, 33.11),
    "A2":      (16.54, 23.39),
    "A3":      (11.69, 16.54),
    "A4":      (8.27,  11.69),
    "A5":      (5.83,   8.27),
    "A6":      (4.13,   5.83),
    "Letter":  (8.5,   11.0),
    "Legal":   (8.5,   14.0),
    "Tabloid": (11.0,  17.0),
}


def rgb_to_hex(rgb):
    r, g, b = [int(round(max(0, min(1, c)) * 255)) for c in rgb]
    return f"#{r:02x}{g:02x}{b:02x}"


def hex_to_rgb(h):
    h = h.lstrip("#")
    if len(h) == 3:
        h = "".join(c * 2 for c in h)
    return tuple(int(h[i:i + 2], 16) / 255.0 for i in (0, 2, 4))


def parse_sym_op(op_str):
    """Parse 'x,y,z' style ops into (rotation 3x3, translation 3)."""
    parts = [p.strip().lower() for p in op_str.split(",")]
    if len(parts) != 3:
        raise ValueError(f"Bad symmetry operation: {op_str}")
    R = np.zeros((3, 3))
    t = np.zeros(3)
    var_idx = {"x": 0, "y": 1, "z": 2}
    for i, expr in enumerate(parts):
        expr = expr.replace(" ", "")
        tokens = []
        cur = ""
        for ch in expr:
            if ch in "+-" and cur:
                tokens.append(cur)
                cur = ch
            else:
                cur += ch
        if cur:
            tokens.append(cur)
        for tok in tokens:
            sign = 1.0
            if tok.startswith("+"):
                tok = tok[1:]
            elif tok.startswith("-"):
                sign = -1.0
                tok = tok[1:]
            if tok in var_idx:
                R[i, var_idx[tok]] = sign
            elif tok and tok[-1] in var_idx and (
                    tok[:-1] == "" or all(c in "0123456789./" for c in tok[:-1])):
                coef = tok[:-1]
                v = tok[-1]
                if "/" in coef:
                    num, den = coef.split("/")
                    val = float(num) / float(den)
                else:
                    val = float(coef) if coef else 1.0
                R[i, var_idx[v]] = sign * val
            else:
                if "/" in tok:
                    num, den = tok.split("/")
                    val = float(num) / float(den)
                else:
                    val = float(tok)
                t[i] += sign * val
    return R, t


def cell_to_params(cell):
    cell = np.array(cell, dtype=float)
    a, b, c = (np.linalg.norm(cell[i]) for i in range(3))
    def ang(u, v):
        nu, nv = np.linalg.norm(u), np.linalg.norm(v)
        if nu < 1e-9 or nv < 1e-9:
            return 90.0
        cosang = np.dot(u, v) / (nu * nv)
        return float(np.degrees(np.arccos(np.clip(cosang, -1, 1))))
    alpha = ang(cell[1], cell[2])
    beta = ang(cell[0], cell[2])
    gamma = ang(cell[0], cell[1])
    return a, b, c, alpha, beta, gamma


def params_to_cell(a, b, c, alpha, beta, gamma):
    al, be, ga = (np.radians(x) for x in (alpha, beta, gamma))
    va = np.array([a, 0.0, 0.0])
    vb = np.array([b * np.cos(ga), b * np.sin(ga), 0.0])
    cx = c * np.cos(be)
    cy = c * (np.cos(al) - np.cos(be) * np.cos(ga)) / max(np.sin(ga), 1e-9)
    cz = np.sqrt(max(c * c - cx * cx - cy * cy, 0.0))
    vc = np.array([cx, cy, cz])
    return np.array([va, vb, vc])


# ===========================================================================
# Core model + rendering (widget-independent, so it is unit-testable)
# ===========================================================================
class CrystalModel:
    """Holds the structure and all display settings, and knows how to render
    a matplotlib figure. No widgets here — this part is testable headlessly."""

    def __init__(self):
        self.atoms = None
        self.original_atoms = None
        self.atom_colors = {}
        self.bond_cutoff = 2.5
        self.pair_cutoffs = {}            # (elemA, elemB) sorted -> Å
        self.show_bonds = True
        self.show_cell = True
        self.show_legend = True
        self.show_axes = True
        self.show_cell_dim = False
        self.perspective = False          # orthographic by default
        self.depthshade = False           # opaque atoms by default
        self.atom_size_scale = 300.0
        self.elev = 25.0
        self.azim = -60.0
        self.zoom = 1.0
        self.bg_color = (1.0, 1.0, 1.0)
        self.image_lo = [0.0, 0.0, 0.0]   # fractional cells appended (-a,-b,-c)
        self.image_hi = [0.0, 0.0, 0.0]   # fractional cells appended (+a,+b,+c)
        self.dim_label_offset = 0.30
        self.cell_dim_style = "corner"   # "corner" (fixed, always clear) or "3d" (near box)
        self.cell_dim_corner = "lower left"
        self.cell_dim_corner_dx = 0.0    # fine nudge on top of the corner preset
        self.legend_corner = "upper right"
        self.legend_dx = 0.0
        self.legend_dy = 0.0
        self.compass_corner = "lower right"
        self.compass_dx = 0.0
        self.compass_dy = 0.0
        self.cell_dim_corner_dy = 0.0    # (axes-fraction units)
        self.show_compass = False
        self.compass_labels = "abc"      # "abc" (lattice vectors) or "xyz" (Cartesian)
        self.proportional_axes = False
        self.box_zoom = 1.1
        self.hide_degenerate_axes = True   # hide ticks of axes nearly edge-on to camera

    # ---- structure I/O ----------------------------------------------------
    def load_from_bytes(self, data, filename):
        suffix = os.path.splitext(filename)[1] or ".cif"
        # Some formats (POSCAR/CONTCAR) have no extension
        base = os.path.basename(filename).upper()
        if suffix == "" and ("POSCAR" in base or "CONTCAR" in base):
            suffix = ".vasp"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tf:
            tf.write(data)
            path = tf.name
        try:
            atoms = read(path)
        finally:
            try:
                os.unlink(path)
            except Exception:
                pass
        if atoms.cell.rank == 0:
            pos = atoms.get_positions()
            span = pos.max(0) - pos.min(0) + 5.0
            atoms.set_cell(np.diag(np.maximum(span, 5.0)))
        self.atoms = atoms
        self.original_atoms = atoms.copy()
        self.atom_colors = self._default_colors(atoms)
        self.image_lo = [0.0, 0.0, 0.0]
        self.image_hi = [0.0, 0.0, 0.0]
        return atoms

    def save_to_path(self, path):
        if self.atoms is not None:
            write(path, self.atoms)

    @staticmethod
    def _default_colors(atoms):
        out = {}
        for sym in set(atoms.get_chemical_symbols()):
            Z = atomic_numbers.get(sym, 1)
            out[sym] = tuple(float(x) for x in jmol_colors[Z])
        return out

    # ---- display atoms (with appended images) -----------------------------
    def get_display_atoms(self):
        if self.atoms is None:
            return None, None
        pos = self.atoms.get_positions()
        syms = list(self.atoms.get_chemical_symbols())
        lo = list(self.image_lo)
        hi = list(self.image_hi)
        eps = 1e-6
        if all(abs(v) < eps for v in lo + hi):
            return pos.copy(), syms
        cell = np.array(self.atoms.cell)
        try:
            inv = np.linalg.inv(cell)
        except np.linalg.LinAlgError:
            return pos.copy(), syms
        base_frac = pos @ inv
        ranges = []
        for v_lo, v_hi in zip(lo, hi):
            i_lo = int(np.floor(-v_lo - eps))
            i_hi = int(np.ceil(v_hi + eps))
            ranges.append(range(i_lo, i_hi + 1))
        kept_pos, kept_syms = [], []
        for i in ranges[0]:
            for j in ranges[1]:
                for k in ranges[2]:
                    nf = base_frac + np.array([i, j, k])
                    mask = (
                        (nf[:, 0] >= -lo[0] - eps) & (nf[:, 0] <= 1 + hi[0] + eps) &
                        (nf[:, 1] >= -lo[1] - eps) & (nf[:, 1] <= 1 + hi[1] + eps) &
                        (nf[:, 2] >= -lo[2] - eps) & (nf[:, 2] <= 1 + hi[2] + eps)
                    )
                    if not mask.any():
                        continue
                    kept_pos.append(nf[mask] @ cell)
                    kept_syms.extend(syms[idx] for idx in np.where(mask)[0])
        if not kept_pos:
            return pos.copy(), syms
        return np.vstack(kept_pos), kept_syms

    # ---- coordination -----------------------------------------------------
    def compute_coordination(self, positions=None, symbols=None):
        if positions is None or symbols is None:
            positions, symbols = self.get_display_atoms()
        if positions is None:
            return np.zeros(0, dtype=int)
        pos = np.asarray(positions)
        syms = list(symbols)
        n = len(pos)
        coord = np.zeros(n, dtype=int)
        if n == 0 or n > 4000:
            return coord
        default_sq = self.bond_cutoff ** 2
        pair_sq = {k: v * v for k, v in self.pair_cutoffs.items()}
        cap_sq = max([default_sq] + list(pair_sq.values()))
        for i in range(n):
            di = pos - pos[i]
            d2 = np.einsum("ij,ij->i", di, di)
            for j in range(n):
                if j == i or d2[j] > cap_sq:
                    continue
                key = tuple(sorted([syms[i], syms[j]]))
                if d2[j] <= pair_sq.get(key, default_sq):
                    coord[i] += 1
        return coord

    # ---- structure edits --------------------------------------------------
    def apply_cell_params(self, a, b, c, alpha, beta, gamma, scale_positions=True):
        if self.atoms is None:
            return
        new_cell = params_to_cell(a, b, c, alpha, beta, gamma)
        if scale_positions:
            frac = self.atoms.get_scaled_positions()
            self.atoms.set_cell(new_cell)
            self.atoms.set_scaled_positions(frac)
        else:
            self.atoms.set_cell(new_cell)

    def build_supercell(self, nx, ny, nz):
        if self.atoms is None:
            return
        self.atoms = self.atoms * (int(nx), int(ny), int(nz))

    def bake_appended(self):
        pos, syms = self.get_display_atoms()
        if pos is None:
            return 0
        cell = np.array(self.atoms.cell)
        self.atoms = Atoms(symbols=syms, positions=pos, cell=cell, pbc=True)
        self.image_lo = [0.0, 0.0, 0.0]
        self.image_hi = [0.0, 0.0, 0.0]
        return len(syms)

    def delete_display_indices(self, idx):
        """Delete atoms by DISPLAY index (bakes appended atoms first)."""
        pos, syms = self.get_display_atoms()
        if pos is None:
            return
        idx = set(int(i) for i in idx)
        keep = [i for i in range(len(syms)) if i not in idx]
        cell = np.array(self.atoms.cell)
        self.atoms = Atoms(symbols=[syms[i] for i in keep],
                           positions=pos[keep], cell=cell, pbc=True)
        self.image_lo = [0.0, 0.0, 0.0]
        self.image_hi = [0.0, 0.0, 0.0]

    def delete_element(self, el):
        pos, syms = self.get_display_atoms()
        if pos is None:
            return 0
        keep = [i for i in range(len(syms)) if syms[i] != el]
        removed = len(syms) - len(keep)
        cell = np.array(self.atoms.cell)
        self.atoms = Atoms(symbols=[syms[i] for i in keep],
                           positions=pos[keep], cell=cell, pbc=True)
        self.image_lo = [0.0, 0.0, 0.0]
        self.image_hi = [0.0, 0.0, 0.0]
        return removed

    def reset_to_original(self):
        if self.original_atoms is None:
            return
        self.atoms = self.original_atoms.copy()
        self.atom_colors = self._default_colors(self.atoms)
        self.image_lo = [0.0, 0.0, 0.0]
        self.image_hi = [0.0, 0.0, 0.0]

    def apply_manual_symmetry(self, ops_text):
        if self.atoms is None:
            return
        ops = []
        for line in ops_text.splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                ops.append(parse_sym_op(line))
        if not ops:
            return
        frac = self.atoms.get_scaled_positions(wrap=True)
        syms = self.atoms.get_chemical_symbols()
        new_frac, new_sym = [], []
        tol = 1e-4
        for f, s in zip(frac, syms):
            for R, t in ops:
                nf = (R @ f + t) % 1.0
                dup = any(np.linalg.norm((nf - g + 0.5) % 1.0 - 0.5) < tol
                          and ns == s for g, ns in zip(new_frac, new_sym))
                if not dup:
                    new_frac.append(nf)
                    new_sym.append(s)
        self.atoms = Atoms(symbols=new_sym, scaled_positions=new_frac,
                           cell=self.atoms.cell, pbc=True)

    def detect_symmetry(self, symprec=0.01):
        if self.atoms is None or not HAS_SPACEGROUP:
            return None
        try:
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sg = get_spacegroup(self.atoms, symprec=symprec)
            return sg
        except Exception:
            return None

    # ---- rendering --------------------------------------------------------
    @staticmethod
    def _points_per_angstrom(ax):
        try:
            fig = ax.get_figure()
            dpi = fig.get_dpi()
            xlim = ax.get_xlim3d(); ylim = ax.get_ylim3d(); zlim = ax.get_zlim3d()
            cx = 0.5 * (xlim[0] + xlim[1]); cy = 0.5 * (ylim[0] + ylim[1])
            cz = 0.5 * (zlim[0] + zlim[1])
            M = ax.get_proj()
            def disp(x, y, z):
                xs, ys, _ = proj3d.proj_transform(x, y, z, M)
                return np.array(ax.transData.transform((xs, ys)))
            p0 = disp(cx, cy, cz)
            px = np.hypot(*(disp(cx + 1, cy, cz) - p0))
            py = np.hypot(*(disp(cx, cy + 1, cz) - p0))
            pz = np.hypot(*(disp(cx, cy, cz + 1) - p0))
            val = np.mean([px, py, pz]) * 72.0 / dpi
            if val > 1e-3:
                return float(val)
        except Exception:
            pass
        return 6.0

    def _set_aspect(self, ax, positions, cell, zoom):
        cell = np.array(cell)
        pts = np.vstack([positions, np.zeros((1, 3))])
        for i, j, k in product([0, 1], repeat=3):
            pts = np.vstack([pts, i * cell[0] + j * cell[1] + k * cell[2]])
        mn = pts.min(0); mx = pts.max(0); center = (mx + mn) / 2
        bz = self.box_zoom
        if self.proportional_axes:
            ranges = np.maximum(mx - mn, 1e-3)
            half = ranges * 0.55 / max(zoom, 0.01)
            ax.set_xlim(center[0]-half[0], center[0]+half[0])
            ax.set_ylim(center[1]-half[1], center[1]+half[1])
            ax.set_zlim(center[2]-half[2], center[2]+half[2])
            try:
                ax.set_box_aspect(ranges, zoom=bz)
            except TypeError:
                ax.set_box_aspect(ranges)
        else:
            span = (mx - mn).max() / 2 * 1.1 / max(zoom, 0.01)
            ax.set_xlim(center[0]-span, center[0]+span)
            ax.set_ylim(center[1]-span, center[1]+span)
            ax.set_zlim(center[2]-span, center[2]+span)
            try:
                ax.set_box_aspect((1, 1, 1), zoom=bz)
            except TypeError:
                ax.set_box_aspect((1, 1, 1))

    def _draw_cell_edges(self, ax, cell):
        cell = np.array(cell)
        corners = [np.zeros(3) + i*cell[0] + j*cell[1] + k*cell[2]
                   for i, j, k in product([0, 1], repeat=3)]
        corners = np.array(corners)
        edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),
                 (3,7),(4,5),(4,6),(5,7),(6,7)]
        for u, v in edges:
            xs, ys, zs = zip(corners[u], corners[v])
            ax.plot(xs, ys, zs, color="black", linewidth=0.8, alpha=0.7)

    def _draw_cell_dim_corner_box(self, ax, cell):
        """Print a, b, c together as one fixed text block in a panel corner
        (axes-fraction coordinates), completely outside the 3-D data — this
        can never end up tangled in atoms/bonds no matter how large or
        dense the displayed structure is, unlike labels anchored near the
        cell edges in 3-D space."""
        cell = np.array(cell)
        lines = []
        for vec, label in zip(cell, ("a", "b", "c")):
            length = float(np.linalg.norm(vec))
            if length > 0:
                lines.append(f"{label} = {length:.3f} Å")
        if not lines:
            return
        text = "\n".join(lines)
        pos = {
            "lower left":  (0.03, 0.03, "left",  "bottom"),
            "lower right": (0.97, 0.03, "right", "bottom"),
            "upper left":  (0.03, 0.97, "left",  "top"),
            "upper right": (0.97, 0.97, "right", "top"),
        }.get(self.cell_dim_corner, (0.03, 0.03, "left", "bottom"))
        x, y, ha, va = pos
        # Fine nudge on top of the corner preset, clamped so it can't be
        # dragged completely off the panel.
        x = float(np.clip(x + self.cell_dim_corner_dx, 0.0, 1.0))
        y = float(np.clip(y + self.cell_dim_corner_dy, 0.0, 1.0))
        ax.text2D(x, y, text, transform=ax.transAxes,
                  fontsize=9, color="#1f4e8c", weight="bold", ha=ha, va=va,
                  bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                           edgecolor="#888888", alpha=0.85, linewidth=0.8))

    def _draw_cell_dim_labels(self, ax, cell, elev, azim):
        cell = np.array(cell)
        cell_vecs = [cell[0], cell[1], cell[2]]
        centroid = 0.5 * (cell_vecs[0] + cell_vecs[1] + cell_vecs[2])
        base_idx = [(0,0),(1,0),(0,1),(1,1)]
        labels = ("a", "b", "c")
        info = []
        for dir_i, (vec, label) in enumerate(zip(cell_vecs, labels)):
            length = float(np.linalg.norm(vec))
            if length <= 0:
                continue
            others = [d for d in (0,1,2) if d != dir_i]
            voa, vob = cell_vecs[others[0]], cell_vecs[others[1]]
            cands = []
            for ia, ib in base_idx:
                base_corner = ia*voa + ib*vob
                mid = base_corner + vec/2
                cands.append((np.linalg.norm(mid-centroid), base_corner))
            cands.sort(key=lambda t: t[0], reverse=True)
            base = cands[0][1]
            mid = base + vec/2
            eu = vec/length
            away = mid - centroid
            perp = away - np.dot(away, eu)*eu
            pn = np.linalg.norm(perp)
            if pn < 1e-6:
                fb = voa - np.dot(voa, eu)*eu
                pn = np.linalg.norm(fb)
                perp = fb if pn > 1e-6 else np.array([0, 0, 1.0])
                pn = max(pn, 1e-6)
            outward = perp/pn
            tp = mid + outward*self.dim_label_offset*length
            info.append([label, length, np.array(tp, float)])

        # de-overlap in screen space using the real projection
        diag = float(np.linalg.norm(cell[0]+cell[1]+cell[2]))
        def proj2d(p):
            if HAS_PROJ3D:
                try:
                    xs, ys, _ = proj3d.proj_transform(p[0], p[1], p[2], ax.get_proj())
                    x2, y2 = ax.transData.transform((xs, ys))
                    return np.array(ax.transAxes.inverted().transform((x2, y2)))
                except Exception:
                    pass
            e = np.radians(elev); az = np.radians(azim)
            u = np.array([-np.sin(az), np.cos(az), 0.0])
            v = np.array([-np.sin(e)*np.cos(az), -np.sin(e)*np.sin(az), np.cos(e)])
            return np.array([p @ u, p @ v]) / max(diag, 1.0)
        min_sep = 0.18
        step = 0.15 * max(diag, 1.0)
        push_dirs = []
        for dx in (-1,0,1):
            for dy in (-1,0,1):
                for dz in (-1,0,1):
                    if dx or dy or dz:
                        vv = np.array([dx,dy,dz], float)
                        push_dirs.append(vv/np.linalg.norm(vv))
        def best_push(pm, pf):
            cur = np.linalg.norm(proj2d(pm)-proj2d(pf))
            bd, bg = None, 0.0
            for d in push_dirs:
                g = np.linalg.norm(proj2d(pm+d*step)-proj2d(pf)) - cur
                if g > bg:
                    bg, bd = g, d*step
            return bd
        for _ in range(20):
            moved = False
            for ii in range(len(info)):
                for jj in range(ii+1, len(info)):
                    if np.linalg.norm(proj2d(info[ii][2])-proj2d(info[jj][2])) < min_sep:
                        d = best_push(info[ii][2], info[jj][2])
                        if d is not None:
                            info[ii][2] = info[ii][2] + d; moved = True
            if not moved:
                break
        # clamp inside axis box and draw
        try:
            xl, yl, zl = ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()
            def clamp(v, lo, hi):
                pad = 0.02*(hi-lo)
                return min(max(v, lo+pad), hi-pad)
        except Exception:
            xl = None
        for label, length, tp in info:
            p = np.array(tp, float)
            if xl is not None:
                p[0] = clamp(p[0], *xl); p[1] = clamp(p[1], *yl); p[2] = clamp(p[2], *zl)
            ax.text(p[0], p[1], p[2], f"{label} = {length:.3f} Å",
                    fontsize=9, color="#1f4e8c", weight="bold",
                    ha="center", va="center")

    def _draw_bonds(self, ax, positions, symbols):
        n = len(positions)
        if n > 2000:
            return
        default_sq = self.bond_cutoff ** 2
        pair_sq = {k: v*v for k, v in self.pair_cutoffs.items()}
        cap_sq = max([default_sq] + list(pair_sq.values()))
        for i in range(n):
            for j in range(i+1, n):
                d = positions[j]-positions[i]
                d2 = float(d @ d)
                if d2 > cap_sq:
                    continue
                key = tuple(sorted([symbols[i], symbols[j]]))
                if d2 <= pair_sq.get(key, default_sq):
                    mid = (positions[i]+positions[j])/2
                    ci = self.atom_colors.get(symbols[i], (.5,.5,.5))
                    cj = self.atom_colors.get(symbols[j], (.5,.5,.5))
                    ax.plot([positions[i,0],mid[0]],[positions[i,1],mid[1]],
                            [positions[i,2],mid[2]], color=ci, linewidth=2)
                    ax.plot([mid[0],positions[j,0]],[mid[1],positions[j,1]],
                            [mid[2],positions[j,2]], color=cj, linewidth=2)

    @staticmethod
    def _hide_degenerate_axis_ticks(ax, elev, azim, threshold=0.92):
        """Hide ticks/labels of any world axis aligned with the view
        direction. Looking straight down an axis collapses its tick marks
        onto a single screen point, which then overlaps every other label —
        this is what turns a top-down or edge-on view into unreadable mush."""
        e = np.radians(elev); a = np.radians(azim)
        view_dir = np.array([np.cos(e)*np.cos(a), np.cos(e)*np.sin(a), np.sin(e)])
        if abs(view_dir[0]) > threshold:
            ax.set_xticks([]); ax.set_xticklabels([]); ax.set_xlabel("")
        if abs(view_dir[1]) > threshold:
            ax.set_yticks([]); ax.set_yticklabels([]); ax.set_ylabel("")
        if abs(view_dir[2]) > threshold:
            ax.set_zticks([]); ax.set_zticklabels([]); ax.set_zlabel("")

    @staticmethod
    def _corner_anchor(corner, dx, dy):
        """Map a corner name to its nominal (x, y) anchor in axes-fraction
        coordinates, plus a fine dx/dy offset on top. Shared by legend and
        compass positioning so 'move it by some value' works the same way
        everywhere in the app."""
        nominal = {
            "lower left":  (0.0, 0.0),
            "lower right": (1.0, 0.0),
            "upper left":  (0.0, 1.0),
            "upper right": (1.0, 1.0),
        }.get(corner, (1.0, 1.0))
        return (nominal[0] + dx, nominal[1] + dy)

    def _draw_compass(self, fig, parent_ax, elev, azim, perspective):
        """Add a tiny corner inset showing the direction of a, b, c (or x,y,z).
        Uses the parent axes' own position to place itself, so it works the
        same whether the parent came from add_subplot or an explicit rect."""
        if self.atoms is None:
            return
        bbox = parent_ax.get_position()
        sz = 0.16  # 16 % of the parent axes
        w = bbox.width * sz
        h = bbox.height * sz
        # Corner choice + fine dx/dy offset (in fractions of the PARENT
        # axes' own width/height, so a value like 0.1 moves it by 10% of
        # the panel regardless of the panel's absolute size on the page).
        ax_frac_x, ax_frac_y = self._corner_anchor(
            self.compass_corner, self.compass_dx, self.compass_dy)
        left = bbox.x0 + ax_frac_x * bbox.width - (w if ax_frac_x > 0.5 else 0)
        bottom = bbox.y0 + ax_frac_y * bbox.height - (h if ax_frac_y > 0.5 else 0)
        try:
            inset = fig.add_axes([left, bottom, w, h], projection="3d")
        except Exception:
            return
        try:
            inset.set_proj_type("persp" if perspective else "ortho")
        except Exception:
            pass
        if self.compass_labels == "abc":
            cell = np.array(self.atoms.cell)
            norms = np.linalg.norm(cell, axis=1, keepdims=True)
            norms[norms < 1e-9] = 1.0
            vecs = cell / norms
            labels = ("a", "b", "c")
        else:
            vecs = np.eye(3)
            labels = ("x", "y", "z")
        colors = ("#d62728", "#2ca02c", "#1f77b4")
        for vec, lbl, col in zip(vecs, labels, colors):
            inset.quiver(0, 0, 0, vec[0], vec[1], vec[2],
                         color=col, arrow_length_ratio=0.25, linewidth=2.0)
            tp = vec * 1.35
            inset.text(tp[0], tp[1], tp[2], lbl, color=col, fontsize=10,
                       weight="bold", ha="center", va="center")
        inset.set_xlim(-1.4, 1.4); inset.set_ylim(-1.4, 1.4); inset.set_zlim(-1.4, 1.4)
        inset.set_axis_off()
        try:
            inset.set_box_aspect((1, 1, 1))
        except Exception:
            pass
        inset.view_init(elev=elev, azim=azim)
        inset.patch.set_alpha(0.0)
        try:
            inset.set_navigate(False)
        except Exception:
            pass

    def render_axes(self, ax, elev, azim, zoom, show_legend, tick_fontsize=None):
        """Draw the full scene into a prepared 3-D axes."""
        positions, symbols = self.get_display_atoms()
        try:
            ax.set_proj_type("persp" if self.perspective else "ortho")
        except Exception:
            pass
        if self.show_cell:
            self._draw_cell_edges(ax, self.atoms.cell)
        ax.view_init(elev=elev, azim=azim)
        self._set_aspect(ax, positions, self.atoms.cell, zoom)
        try:
            ax.get_figure().canvas.draw()
        except Exception:
            pass
        ppa = self._points_per_angstrom(ax)
        size_k = (ppa**2) * (self.atom_size_scale/300.0) * 0.20
        unique = sorted(set(symbols), key=lambda s: atomic_numbers.get(s, 0))
        handles = []
        for el in unique:
            mask = np.array([s == el for s in symbols])
            p = positions[mask]
            color = self.atom_colors.get(el, (.5,.5,.5))
            Z = atomic_numbers.get(el, 1)
            size = max((covalent_radii[Z]**2)*size_k, 1.0)
            ax.scatter(p[:,0], p[:,1], p[:,2], s=size, c=[color],
                       edgecolors="black", linewidths=0.5,
                       depthshade=self.depthshade, label=el)
            handles.append(Patch(facecolor=color, edgecolor="black", label=el))
        if self.show_bonds and self.bond_cutoff > 0:
            self._draw_bonds(ax, positions, symbols)
        ax.set_facecolor(self.bg_color)
        if not self.show_axes:
            ax.set_axis_off()
        else:
            ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
            if tick_fontsize:
                ax.tick_params(labelsize=tick_fontsize)
                for lbl_ax in (ax.xaxis, ax.yaxis, ax.zaxis):
                    lbl_ax.label.set_fontsize(tick_fontsize + 1)
            # Auto-hide ticks for any axis the camera is looking straight
            # down — otherwise their tick labels collapse onto one point
            # and pile up on top of the other axes' labels.
            if self.hide_degenerate_axes:
                self._hide_degenerate_axis_ticks(ax, elev, azim)
        if self.show_cell and self.show_cell_dim:
            if self.cell_dim_style == "corner":
                self._draw_cell_dim_corner_box(ax, self.atoms.cell)
            else:
                self._draw_cell_dim_labels(ax, self.atoms.cell, elev, azim)
        if show_legend and handles:
            ax.legend(handles=handles, loc=self.legend_corner, fontsize=9,
                     framealpha=0.85,
                     bbox_to_anchor=self._corner_anchor(
                         self.legend_corner, self.legend_dx, self.legend_dy))
        if self.show_compass:
            self._draw_compass(ax.get_figure(), ax, elev, azim, self.perspective)

    def make_live_figure(self, figsize=(7, 6), dpi=100):
        fig = plt.figure(figsize=figsize, dpi=dpi, facecolor=self.bg_color)
        if self.atoms is None:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "Upload a structure to begin",
                    ha="center", va="center", fontsize=14)
            ax.set_axis_off()
            return fig
        if self.show_compass:
            # Give the main axes its FINAL position up front (a fixed,
            # generous margin) before rendering, so when render_axes places
            # the compass inset relative to the main axes' bounding box,
            # that box is already final. Calling tight_layout() AFTER the
            # compass is placed would move the main axes again and leave
            # the inset hugging stale, now-wrong coordinates.
            ax = fig.add_axes([0.10, 0.10, 0.82, 0.82], projection="3d")
        else:
            ax = fig.add_subplot(111, projection="3d")
        self.render_axes(ax, self.elev, self.azim, self.zoom, self.show_legend)
        if not self.show_compass:
            try:
                fig.tight_layout()
            except Exception:
                pass
        return fig

    @staticmethod
    def panel_rects(page, rows, cols, margin, wspace, hspace, has_title):
        """Compute explicit [left, bottom, width, height] rects (figure
        fraction) for each panel, top row first, left-to-right.

        Root cause this works around: matplotlib's Axes3D with a cubic box
        aspect always renders as a SQUARE (a cube's orthographic silhouette
        is square from any angle), regardless of the rectangle you give it.
        Handing it a tall rectangle — which happens automatically whenever
        the page is portrait — makes it center a shrunk square inside that
        rectangle, and the unused space above/below is exactly the huge gap
        that used to appear between panel rows. The fix is to allocate a
        rectangle that is ALREADY square in real inches (the limiting
        dimension of each grid cell), then re-center the whole panel block
        on the page so the leftover space becomes ordinary outer margin
        instead of a gap buried between rows or columns.
        """
        pw, ph = page
        left_m = margin
        right_m = margin
        bottom_m = margin
        top_m = margin
        if has_title:
            top_m += 0.05 * ph  # reserve a strip for fig.suptitle, in inches
        usable_w_in = max(pw - left_m - right_m, 0.1)
        usable_h_in = max(ph - top_m - bottom_m, 0.1)
        col_w_in = usable_w_in / (cols + (cols - 1) * max(wspace, -0.9))
        row_h_in = usable_h_in / (rows + (rows - 1) * max(hspace, -0.9))
        # The panel itself always renders square (cubic box aspect), so size
        # every cell to the smaller of the two — the larger dimension would
        # just get silently shrunk back down anyway.
        panel_in = max(min(col_w_in, row_h_in), 0.1)
        gap_w_in = wspace * panel_in
        gap_h_in = hspace * panel_in
        block_w_in = cols * panel_in + (cols - 1) * gap_w_in
        block_h_in = rows * panel_in + (rows - 1) * gap_h_in
        # Centre the whole block in the usable area so leftover space (from
        # a grid aspect ratio that doesn't match the page) becomes symmetric
        # outer margin, not a lump stuck between two rows/columns.
        start_left_in = left_m + max(usable_w_in - block_w_in, 0) / 2
        start_top_in = top_m + max(usable_h_in - block_h_in, 0) / 2
        rects = []
        for r in range(rows):
            for c in range(cols):
                left_in = start_left_in + c * (panel_in + gap_w_in)
                top_in = start_top_in + r * (panel_in + gap_h_in)
                bottom_in = ph - top_in - panel_in
                rects.append([left_in / pw, bottom_in / ph,
                             panel_in / pw, panel_in / ph])
        return rects

    def make_pdf_figure(self, page, rows, cols, margin, wspace, hspace,
                        panels, main_title=""):
        """panels: list of dicts with keys caption, elev, azim, zoom, legend."""
        fig = plt.figure(figsize=page, facecolor=self.bg_color)
        if main_title:
            fig.suptitle(main_title, fontsize=14)
        rects = self.panel_rects(page, rows, cols, margin, wspace, hspace,
                                 bool(main_title))
        # Smaller panels need smaller tick/axis-label fonts or the text
        # itself becomes the thing that overlaps. Scale from the panel's
        # physical size on the page.
        panel_w_in = rects[0][2] * page[0] if rects else page[0]
        panel_h_in = rects[0][3] * page[1] if rects else page[1]
        tick_fs = float(np.clip(min(panel_w_in, panel_h_in) * 2.4, 6.5, 11))
        for i in range(rows*cols):
            ax = fig.add_axes(rects[i], projection="3d")
            if i < len(panels):
                pd = panels[i]
            else:
                pd = dict(caption="", elev=self.elev, azim=self.azim,
                          zoom=self.zoom, legend=False)
            # Per-panel overrides for cell-dimension labels and the compass:
            # temporarily swap the global setting, render, then restore —
            # this is what makes "show cell dims only on this one panel"
            # actually work, instead of it applying to every panel.
            saved_celldim = self.show_cell_dim
            saved_compass = self.show_compass
            if "celldim" in pd:
                self.show_cell_dim = pd["celldim"]
            if "compass" in pd:
                self.show_compass = pd["compass"]
            try:
                self.render_axes(ax, pd["elev"], pd["azim"], pd["zoom"], pd["legend"],
                                 tick_fontsize=tick_fs)
            finally:
                self.show_cell_dim = saved_celldim
                self.show_compass = saved_compass
            if pd.get("caption"):
                ax.set_title(pd["caption"], fontsize=pd.get("caption_size", 12))
        return fig


# ===========================================================================
# ipywidgets UI
# ===========================================================================
class CrystalVisualizerColab:
    def __init__(self):
        self.m = CrystalModel()
        self._suspend = False
        self.color_pickers = {}
        self.pair_rows = []           # list of (elem1, elem2, floattext) widgets
        self.panel_widgets = []       # per-panel control dicts
        self._build_ui()

    # ---- small helpers ----------------------------------------------------
    def _render(self, *_):
        if self._suspend:
            return
        with self.out:
            clear_output(wait=True)
            fig = self.m.make_live_figure()
            display(fig)
            plt.close(fig)

    def _status(self, msg):
        self.status.value = f"<i>{msg}</i>"

    # ---- UI construction --------------------------------------------------
    def _build_ui(self):
        self.out = widgets.Output(layout=widgets.Layout(
            border="1px solid #ccc", min_height="480px"))
        self.status = widgets.HTML("<i>Upload a crystallographic file to begin.</i>")

        tabs = widgets.Tab()
        tabs.children = [
            self._tab_file(), self._tab_display(), self._tab_atoms(),
            self._tab_symmetry(), self._tab_export(),
        ]
        for i, name in enumerate(["File / Cell", "Display", "Atoms",
                                  "Symmetry", "Export"]):
            tabs.set_title(i, name)

        self.ui = widgets.VBox([
            widgets.HTML("<h3>Crystal Structure Visualizer "
                         "<small>(Colab edition)</small></h3>"),
            tabs, self.status, self.out,
        ])

    # ---------- File / Cell tab -------------------------------------------
    def _tab_file(self):
        self.w_upload = widgets.FileUpload(
            accept=".cif,.vasp,.poscar,.xyz,.xsf,.cube,.pdb,POSCAR,CONTCAR",
            multiple=False, description="Upload")
        self.w_upload.observe(self._on_upload, names="value")

        self.w_save = widgets.Button(description="Download current (.cif)",
                                     icon="download")
        self.w_save.on_click(self._on_save)
        self.w_reset = widgets.Button(description="Reset to loaded file",
                                     icon="undo")
        self.w_reset.on_click(self._on_reset)

        # cell parameters
        self.w_a = widgets.FloatText(description="a (Å)")
        self.w_b = widgets.FloatText(description="b (Å)")
        self.w_c = widgets.FloatText(description="c (Å)")
        self.w_al = widgets.FloatText(description="α (°)")
        self.w_be = widgets.FloatText(description="β (°)")
        self.w_ga = widgets.FloatText(description="γ (°)")
        self.w_scale_pos = widgets.Checkbox(value=True,
                                            description="Keep fractional coords")
        w_apply_cell = widgets.Button(description="Apply cell", icon="check")
        w_apply_cell.on_click(self._on_apply_cell)

        # supercell
        self.w_nx = widgets.BoundedIntText(value=1, min=1, max=20, description="nx")
        self.w_ny = widgets.BoundedIntText(value=1, min=1, max=20, description="ny")
        self.w_nz = widgets.BoundedIntText(value=1, min=1, max=20, description="nz")
        w_super = widgets.Button(description="Build supercell", icon="th")
        w_super.on_click(self._on_supercell)

        # append cells (fractional)
        def mk(desc):
            return widgets.BoundedFloatText(value=0.0, min=0.0, max=5.0,
                                            step=0.1, description=desc,
                                            layout=widgets.Layout(width="140px"))
        self.w_ax_lo, self.w_ax_hi = mk("a −"), mk("a +")
        self.w_by_lo, self.w_by_hi = mk("b −"), mk("b +")
        self.w_cz_lo, self.w_cz_hi = mk("c −"), mk("c +")
        for w in (self.w_ax_lo, self.w_ax_hi, self.w_by_lo, self.w_by_hi,
                  self.w_cz_lo, self.w_cz_hi):
            w.observe(self._on_append_change, names="value")
        w_clear_app = widgets.Button(description="Clear appended")
        w_clear_app.on_click(self._on_clear_append)

        return widgets.VBox([
            widgets.HTML("<b>File</b>"),
            widgets.HBox([self.w_upload, self.w_save, self.w_reset]),
            widgets.HTML("<b>Unit cell</b>"),
            widgets.HBox([self.w_a, self.w_b, self.w_c]),
            widgets.HBox([self.w_al, self.w_be, self.w_ga]),
            widgets.HBox([self.w_scale_pos, w_apply_cell]),
            widgets.HTML("<b>Supercell expansion</b>"),
            widgets.HBox([self.w_nx, self.w_ny, self.w_nz, w_super]),
            widgets.HTML("<b>Append cells around (display only; fractional)</b>"),
            widgets.HBox([self.w_ax_lo, self.w_ax_hi]),
            widgets.HBox([self.w_by_lo, self.w_by_hi]),
            widgets.HBox([self.w_cz_lo, self.w_cz_hi]),
            w_clear_app,
        ])

    # ---------- Display tab -----------------------------------------------
    def _tab_display(self):
        self.w_bond = widgets.BoundedFloatText(
            value=self.m.bond_cutoff, min=0.0, max=20.0, step=0.05,
            description="Max bond (Å)")
        self.w_bond.observe(self._on_setting, names="value")

        # pair cutoffs
        self.pair_box = widgets.VBox([])
        self.w_pair_e1 = widgets.Dropdown(description="Elem 1",
                                          layout=widgets.Layout(width="150px"))
        self.w_pair_e2 = widgets.Dropdown(description="Elem 2",
                                          layout=widgets.Layout(width="150px"))
        self.w_pair_v = widgets.BoundedFloatText(value=2.5, min=0.0, max=50.0,
                                                 step=0.05, description="Max Å",
                                                 layout=widgets.Layout(width="150px"))
        w_add_pair = widgets.Button(description="Add pair cutoff", icon="plus")
        w_add_pair.on_click(self._on_add_pair)

        self.w_legend = widgets.Checkbox(value=True, description="Show legend")
        self.w_cell = widgets.Checkbox(value=True, description="Show unit-cell edges")
        self.w_axes = widgets.Checkbox(value=True, description="Show axes / ticks")
        self.w_celldim = widgets.Checkbox(value=False, description="Show cell dimensions")
        self.w_persp = widgets.Checkbox(value=False, description="Perspective projection")
        self.w_shade = widgets.Checkbox(value=False, description="Depth shading")
        self.w_propax = widgets.Checkbox(value=False, description="Proportional axes")
        self.w_hidedeg = widgets.Checkbox(value=True,
                                          description="Auto-hide edge-on axis ticks")
        for w in (self.w_legend, self.w_cell, self.w_axes, self.w_celldim,
                  self.w_persp, self.w_shade, self.w_propax, self.w_hidedeg):
            w.observe(self._on_setting, names="value")

        self.w_dimstyle = widgets.Dropdown(
            options=[("Corner box (always clear)", "corner"),
                    ("Attached to cell edges (3-D, classic)", "3d")],
            value="corner", description="Dim. style")
        self.w_dimcorner = widgets.Dropdown(
            options=["lower left", "lower right", "upper left", "upper right"],
            value="lower left", description="Corner")
        self.w_dimstyle.observe(self._on_setting, names="value")
        self.w_dimcorner.observe(self._on_setting, names="value")

        self.w_dim_dx = widgets.FloatSlider(value=0.0, min=-0.4, max=0.4, step=0.01,
                                            description="Move X",
                                            continuous_update=False)
        self.w_dim_dy = widgets.FloatSlider(value=0.0, min=-0.4, max=0.4, step=0.01,
                                            description="Move Y",
                                            continuous_update=False)
        self.w_dim_dx.observe(self._on_setting, names="value")
        self.w_dim_dy.observe(self._on_setting, names="value")

        self.w_compass = widgets.Checkbox(value=False,
                                          description="Show orientation compass")
        self.w_compass_lbl = widgets.Dropdown(
            options=[("a, b, c (lattice vectors)", "abc"),
                    ("x, y, z (Cartesian)", "xyz")],
            value="abc", description="Compass labels")
        self.w_compass_corner = widgets.Dropdown(
            options=["lower left", "lower right", "upper left", "upper right"],
            value="lower right", description="Corner")
        self.w_compass_dx = widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01,
                                                description="Move X",
                                                continuous_update=False)
        self.w_compass_dy = widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01,
                                                description="Move Y",
                                                continuous_update=False)
        for w in (self.w_compass, self.w_compass_lbl, self.w_compass_corner,
                 self.w_compass_dx, self.w_compass_dy):
            w.observe(self._on_setting, names="value")

        self.w_legend_corner = widgets.Dropdown(
            options=["lower left", "lower right", "upper left", "upper right"],
            value="upper right", description="Corner")
        self.w_legend_dx = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.01,
                                               description="Move X",
                                               continuous_update=False)
        self.w_legend_dy = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.01,
                                               description="Move Y",
                                               continuous_update=False)
        for w in (self.w_legend_corner, self.w_legend_dx, self.w_legend_dy):
            w.observe(self._on_setting, names="value")

        self.w_size = widgets.FloatSlider(value=300, min=20, max=2000, step=20,
                                          description="Atom size",
                                          continuous_update=False)
        self.w_size.observe(self._on_setting, names="value")
        self.w_boxzoom = widgets.FloatSlider(value=1.1, min=0.5, max=2.5, step=0.05,
                                             description="Panel fill",
                                             continuous_update=False)
        self.w_boxzoom.observe(self._on_setting, names="value")

        # rotation
        self.w_elev = widgets.FloatSlider(value=25, min=-180, max=180, step=5,
                                          description="Elevation",
                                          continuous_update=False)
        self.w_azim = widgets.FloatSlider(value=-60, min=-360, max=360, step=5,
                                          description="Azimuth",
                                          continuous_update=False)
        self.w_zoom = widgets.FloatSlider(value=1.0, min=0.2, max=8.0, step=0.1,
                                          description="Zoom",
                                          continuous_update=False)
        for w in (self.w_elev, self.w_azim, self.w_zoom):
            w.observe(self._on_setting, names="value")

        def preset(e, a):
            def _cb(_):
                self._suspend = True
                self.w_elev.value = e; self.w_azim.value = a
                self._suspend = False
                self._pull_settings(); self._render()
            return _cb
        b_a = widgets.Button(description="a-axis"); b_a.on_click(preset(0, 0))
        b_b = widgets.Button(description="b-axis"); b_b.on_click(preset(0, 90))
        b_c = widgets.Button(description="c-axis"); b_c.on_click(preset(90, -90))
        b_i = widgets.Button(description="iso"); b_i.on_click(preset(25, -60))

        # colours
        self.color_box = widgets.VBox([])

        return widgets.VBox([
            widgets.HTML("<b>Bonds</b>"),
            self.w_bond,
            widgets.HTML("<small>Tip: keep below the interlayer separation so "
                         "van-der-Waals layers stay unbonded.</small>"),
            widgets.HTML("<b>Pair-specific bond cutoffs</b>"),
            widgets.HBox([self.w_pair_e1, self.w_pair_e2, self.w_pair_v, w_add_pair]),
            self.pair_box,
            widgets.HTML("<b>Scene</b>"),
            widgets.HBox([self.w_legend, self.w_cell, self.w_axes]),
            widgets.HBox([self.w_legend_corner, self.w_legend_dx, self.w_legend_dy]),
            widgets.HTML("<small>Move the legend if it happens to cover part "
                         "of the structure — pick a corner, then fine-tune "
                         "with Move X / Move Y.</small>"),
            widgets.HBox([self.w_celldim, self.w_persp, self.w_shade]),
            widgets.HBox([self.w_dimstyle, self.w_dimcorner]),
            widgets.HTML("<small>'Corner box' prints a, b, c together in one "
                         "fixed panel corner, outside the 3-D data — always "
                         "readable, regardless of how large or dense the "
                         "structure is (recommended for supercells / "
                         "appended cells). 'Attached to cell edges' looks "
                         "nicer for a single, uncrowded cell but can get "
                         "tangled in atoms for dense views.</small>"),
            widgets.HBox([self.w_dim_dx, self.w_dim_dy]),
            widgets.HTML("<small>Fine-tune the corner box position (fraction "
                         "of the panel) if it happens to land on a tick "
                         "label or a stray atom in your specific view.</small>"),
            widgets.HTML("<b>Compass</b>"),
            widgets.HBox([self.w_compass, self.w_compass_lbl]),
            widgets.HBox([self.w_compass_corner, self.w_compass_dx, self.w_compass_dy]),
            widgets.HTML("<small>Adds a small inset showing which way a, b, c "
                         "(or x, y, z) point — handy once you've rotated away "
                         "from a standard view. Move it the same way as the "
                         "cell-dimension box if it overlaps something.</small>"),
            widgets.HBox([self.w_propax, self.w_hidedeg]),
            widgets.HTML("<small>'Auto-hide edge-on axis ticks' fixes the mess "
                         "you get from a straight top-down or side-on view, "
                         "where one axis's tick labels collapse onto a single "
                         "point and pile on top of the others.</small>"),
            self.w_size, self.w_boxzoom,
            widgets.HTML("<b>View rotation</b>"),
            self.w_elev, self.w_azim, self.w_zoom,
            widgets.HBox([b_a, b_b, b_c, b_i]),
            widgets.HTML("<b>Per-element colours</b>"),
            self.color_box,
        ])

    # ---------- Atoms tab -------------------------------------------------
    def _tab_atoms(self):
        self.w_atomlist = widgets.SelectMultiple(
            options=[], rows=12,
            layout=widgets.Layout(width="99%"),
            description="")
        w_refresh = widgets.Button(description="Refresh list / sync appended",
                                   icon="refresh")
        w_refresh.on_click(self._on_atoms_refresh)
        w_sel_unconn = widgets.Button(description="Select unconnected (0 bonds)")
        w_sel_unconn.on_click(self._on_select_unconnected)
        w_del_sel = widgets.Button(description="Delete selected", icon="trash",
                                   button_style="danger")
        w_del_sel.on_click(self._on_delete_selected)
        self.w_del_elem = widgets.Dropdown(description="Element",
                                           layout=widgets.Layout(width="180px"))
        w_del_elem_btn = widgets.Button(description="Delete all of element")
        w_del_elem_btn.on_click(self._on_delete_element)
        self.atoms_info = widgets.HTML("")

        return widgets.VBox([
            widgets.HTML("<b>Atoms (includes appended). "
                         "Ctrl/Shift-click for multi-select.</b>"),
            self.w_atomlist,
            widgets.HBox([w_refresh, w_sel_unconn, w_del_sel]),
            widgets.HBox([self.w_del_elem, w_del_elem_btn]),
            self.atoms_info,
        ])

    # ---------- Symmetry tab ----------------------------------------------
    def _tab_symmetry(self):
        self.sym_out = widgets.HTML(
            "Load a structure to detect symmetry." if HAS_SPACEGROUP
            else "Install spglib for symmetry detection.")
        self.w_symprec = widgets.BoundedFloatText(value=0.01, min=1e-5, max=1.0,
                                                  step=0.001, description="Tol (Å)")
        w_detect = widgets.Button(description="Detect symmetry (spglib)")
        w_detect.on_click(self._on_detect_sym)
        self.w_manual = widgets.Textarea(
            value="x,y,z\n-x,-y,-z", description="Ops",
            layout=widgets.Layout(width="99%", height="120px"))
        w_apply_manual = widgets.Button(description="Apply manual operations")
        w_apply_manual.on_click(self._on_apply_manual)
        return widgets.VBox([
            widgets.HTML("<b>Symmetry from file</b>"),
            widgets.HBox([self.w_symprec, w_detect]),
            self.sym_out,
            widgets.HTML("<b>Manual symmetry operations "
                         "(one per line, e.g. -x+1/2,y,-z+1/2)</b>"),
            self.w_manual, w_apply_manual,
        ])

    # ---------- Export tab ------------------------------------------------
    def _tab_export(self):
        self.w_page = widgets.Dropdown(options=list(PAGE_SIZES) + ["Custom"],
                                       value="A4", description="Page")
        self.w_orient = widgets.Dropdown(options=["portrait", "landscape"],
                                         value="portrait", description="Orient")
        self.w_pw = widgets.BoundedFloatText(value=8.27, min=1, max=100, step=0.1,
                                             description="W (in)")
        self.w_ph = widgets.BoundedFloatText(value=11.69, min=1, max=100, step=0.1,
                                             description="H (in)")
        self.w_page.observe(self._on_page_preset, names="value")
        self.w_orient.observe(self._on_page_preset, names="value")
        self.w_margin = widgets.BoundedFloatText(value=0.5, min=0, max=5, step=0.1,
                                                 description="Margin")
        self.w_wspace = widgets.BoundedFloatText(value=0.05, min=-0.9, max=2,
                                                 step=0.01, description="H-gap")
        self.w_hspace = widgets.BoundedFloatText(value=0.1, min=-0.9, max=2,
                                                 step=0.01, description="V-gap")
        self.w_rows = widgets.BoundedIntText(value=1, min=1, max=4, description="Rows")
        self.w_cols = widgets.BoundedIntText(value=1, min=1, max=4, description="Cols")
        self.w_rows.observe(self._on_panel_grid, names="value")
        self.w_cols.observe(self._on_panel_grid, names="value")
        self.w_title = widgets.Text(description="Title",
                                    placeholder="optional figure title")

        self.panel_area = widgets.VBox([])
        self._rebuild_panels()

        w_preview = widgets.Button(description="Preview PDF layout", icon="eye")
        w_preview.on_click(self._on_preview_pdf)
        w_export = widgets.Button(description="Export & download PDF",
                                  icon="download", button_style="success")
        w_export.on_click(self._on_export_pdf)

        return widgets.VBox([
            widgets.HTML("<b>PDF page</b>"),
            widgets.HBox([self.w_page, self.w_orient]),
            widgets.HBox([self.w_pw, self.w_ph, self.w_margin]),
            widgets.HTML("<b>Spacing</b>"),
            widgets.HBox([self.w_wspace, self.w_hspace]),
            widgets.HTML("<b>Panel layout</b>"),
            widgets.HBox([self.w_rows, self.w_cols]),
            widgets.HTML("<small>Set the angle in the <i>Display</i> tab, then "
                         "click a panel's <b>Use current</b> to copy it in — "
                         "or set every panel at once with the button below.</small>"),
            self._make_sync_all_button(),
            self.panel_area,
            self.w_title,
            widgets.HBox([w_preview, w_export]),
        ])

    # =======================================================================
    # Callbacks
    # =======================================================================
    def _on_upload(self, change):
        up = self.w_upload.value
        if not up:
            return
        # ipywidgets v8: tuple of dicts; v7: dict keyed by filename
        items = list(up.values()) if isinstance(up, dict) else list(up)
        if not items:
            return
        item = items[-1]
        name = item.get("name") or "structure.cif"
        content = item["content"]
        data = bytes(content)
        try:
            self.m.load_from_bytes(data, name)
        except Exception as e:
            self._status(f"Read error: {e}")
            return
        self._sync_cell_widgets()
        self._rebuild_colors()
        self._rebuild_pair_dropdowns()
        self._refresh_atom_list()
        self._status(f"Loaded {name} — {len(self.m.atoms)} atoms.")
        self._render()

    def _on_save(self, _):
        if self.m.atoms is None:
            return
        path = "/content/structure.cif" if os.path.isdir("/content") else "structure.cif"
        self.m.save_to_path(path)
        self._status(f"Saved to {path}")
        try:
            from google.colab import files
            files.download(path)
        except Exception:
            self._status(f"Saved to {path} (download only works in Colab).")

    def _on_reset(self, _):
        self.m.reset_to_original()
        self._suspend = True
        self._sync_cell_widgets()
        self._clear_append_widgets()
        self._suspend = False
        self._rebuild_colors()
        self._refresh_atom_list()
        self._status("Reset to loaded file.")
        self._render()

    def _on_apply_cell(self, _):
        if self.m.atoms is None:
            return
        self.m.apply_cell_params(self.w_a.value, self.w_b.value, self.w_c.value,
                                 self.w_al.value, self.w_be.value, self.w_ga.value,
                                 scale_positions=self.w_scale_pos.value)
        self._status("Cell parameters applied.")
        self._render()

    def _on_supercell(self, _):
        if self.m.atoms is None:
            return
        self.m.build_supercell(self.w_nx.value, self.w_ny.value, self.w_nz.value)
        self._sync_cell_widgets()
        self._refresh_atom_list()
        self._rebuild_colors()
        self._status(f"Built {self.w_nx.value}×{self.w_ny.value}×{self.w_nz.value} "
                     f"supercell — {len(self.m.atoms)} atoms.")
        self._render()

    def _on_append_change(self, _):
        if self._suspend:
            return
        self.m.image_lo = [self.w_ax_lo.value, self.w_by_lo.value, self.w_cz_lo.value]
        self.m.image_hi = [self.w_ax_hi.value, self.w_by_hi.value, self.w_cz_hi.value]
        self._refresh_atom_list()
        self._render()

    def _on_clear_append(self, _):
        self._clear_append_widgets()
        self.m.image_lo = [0, 0, 0]; self.m.image_hi = [0, 0, 0]
        self._refresh_atom_list()
        self._render()

    def _on_setting(self, _):
        if self._suspend:
            return
        self._pull_settings()
        self._render()

    def _on_add_pair(self, _):
        e1 = self.w_pair_e1.value; e2 = self.w_pair_e2.value
        if not e1 or not e2:
            return
        key = tuple(sorted([e1, e2]))
        self.m.pair_cutoffs[key] = self.w_pair_v.value
        self._rebuild_pair_rows()
        self._render()

    def _on_atoms_refresh(self, _):
        # "sync appended -> editable": bake appended atoms so they can be deleted
        eps = 1e-6
        if not all(abs(v) < eps for v in self.m.image_lo + self.m.image_hi):
            n = self.m.bake_appended()
            self._clear_append_widgets()
            self._status(f"Appended atoms baked in — {n} atoms now editable.")
        self._refresh_atom_list()
        self._rebuild_colors()
        self._render()

    def _on_select_unconnected(self, _):
        pos, syms = self.m.get_display_atoms()
        if pos is None:
            return
        coord = self.m.compute_coordination(pos, syms)
        unconn = [i for i in range(len(coord)) if coord[i] == 0]
        # options values are display indices
        self.w_atomlist.value = tuple(unconn)
        self._status(f"Selected {len(unconn)} unconnected atom(s).")

    def _on_delete_selected(self, _):
        sel = list(self.w_atomlist.value)
        if not sel:
            self._status("No atoms selected.")
            return
        self.m.delete_display_indices(sel)
        self._clear_append_widgets()
        self._refresh_atom_list()
        self._rebuild_colors()
        self._status(f"Deleted {len(sel)} atom(s).")
        self._render()

    def _on_delete_element(self, _):
        el = self.w_del_elem.value
        if not el:
            return
        removed = self.m.delete_element(el)
        self._clear_append_widgets()
        self._refresh_atom_list()
        self._rebuild_colors()
        self._status(f"Deleted {removed} {el} atom(s).")
        self._render()

    def _on_detect_sym(self, _):
        sg = self.m.detect_symmetry(self.w_symprec.value)
        if sg is None:
            self.sym_out.value = "Symmetry detection unavailable or failed."
        else:
            self.sym_out.value = (f"<b>Space group:</b> {sg.symbol} "
                                  f"(No. {sg.no})")

    def _on_apply_manual(self, _):
        if self.m.atoms is None:
            return
        try:
            self.m.apply_manual_symmetry(self.w_manual.value)
        except Exception as e:
            self._status(f"Symmetry error: {e}")
            return
        self._refresh_atom_list()
        self._rebuild_colors()
        self._status(f"Manual symmetry applied — {len(self.m.atoms)} atoms.")
        self._render()

    def _on_page_preset(self, _):
        name = self.w_page.value
        if name in PAGE_SIZES:
            w, h = PAGE_SIZES[name]
            if self.w_orient.value == "landscape":
                w, h = h, w
            self._suspend = True
            self.w_pw.value = w; self.w_ph.value = h
            self._suspend = False

    def _on_panel_grid(self, _):
        self._rebuild_panels()

    def _make_sync_all_button(self):
        btn = widgets.Button(description="Use current view for ALL panels",
                             button_style="info",
                             layout=widgets.Layout(width="260px"))
        btn.on_click(self._on_sync_all_panels)
        return btn

    def _on_sync_all_panels(self, _):
        for pw in self.panel_widgets:
            pw["elev"].value = self.m.elev
            pw["azim"].value = self.m.azim
            pw["zoom"].value = self.m.zoom
        self._status(f"All panels set to elev={self.m.elev:g}, "
                     f"azim={self.m.azim:g}, zoom={self.m.zoom:g} "
                     f"(the Display tab's current view).")

    def _make_use_current_cb(self, idx):
        def _cb(_):
            pw = self.panel_widgets[idx]
            pw["elev"].value = self.m.elev
            pw["azim"].value = self.m.azim
            pw["zoom"].value = self.m.zoom
            self._status(f"Panel {idx+1} set to the Display tab's current view "
                         f"(elev={self.m.elev:g}, azim={self.m.azim:g}, "
                         f"zoom={self.m.zoom:g}).")
        return _cb

    def _rebuild_panels(self):
        n = self.w_rows.value * self.w_cols.value
        self.panel_widgets = []
        rows_ui = []
        # Number widgets: no built-in description (which eats width and was
        # truncating the value, e.g. "90.00" rendering as "9("). A small
        # fixed-width Label sits alongside instead, so the input box gets
        # its full declared width for the value itself.
        num_layout = widgets.Layout(width="100px")
        lbl_layout = widgets.Layout(width="34px")
        # Checkboxes have the SAME description-clipping problem the numeric
        # fields had: packing a description string into a narrow Checkbox
        # layout squeezes the text down to nothing in Colab's CSS, leaving
        # an unlabeled empty box. Same fix: bare checkbox (no description)
        # plus its own separate, adequately-wide Label.
        chk_box_layout = widgets.Layout(width="24px")
        chk_lbl_layout = widgets.Layout(width="68px")
        for i in range(n):
            cap = widgets.Text(value=f"View {i+1}", description=f"P{i+1}",
                               layout=widgets.Layout(width="200px"))
            elev = widgets.FloatText(value=self.m.elev, description="",
                                     layout=num_layout)
            azim = widgets.FloatText(value=self.m.azim, description="",
                                     layout=num_layout)
            zoom = widgets.FloatText(value=self.m.zoom, description="",
                                     layout=num_layout)
            # Legend always starts OFF for every new panel — a legend is a
            # per-view choice, not something that should follow whatever the
            # global toggle happens to be. This is what makes "check legend
            # for View 2 only" actually mean only View 2.
            leg = widgets.Checkbox(value=False, description="",
                                   indent=False, layout=chk_box_layout)
            celldim = widgets.Checkbox(value=self.m.show_cell_dim, description="",
                                       indent=False, layout=chk_box_layout)
            compass = widgets.Checkbox(value=self.m.show_compass, description="",
                                       indent=False, layout=chk_box_layout)
            use_cur = widgets.Button(description="Use current",
                                     layout=widgets.Layout(width="110px"))
            use_cur.on_click(self._make_use_current_cb(i))
            self.panel_widgets.append(dict(caption=cap, elev=elev, azim=azim,
                                           zoom=zoom, legend=leg,
                                           celldim=celldim, compass=compass))
            rows_ui.append(widgets.VBox([
                widgets.HBox([
                    cap,
                    widgets.Label("elev", layout=lbl_layout), elev,
                    widgets.Label("azim", layout=lbl_layout), azim,
                    widgets.Label("zoom", layout=lbl_layout), zoom,
                    use_cur,
                ]),
                widgets.HBox([
                    leg, widgets.Label("legend", layout=chk_lbl_layout),
                    celldim, widgets.Label("cell dims", layout=chk_lbl_layout),
                    compass, widgets.Label("compass", layout=chk_lbl_layout),
                ]),
            ]))
        self.panel_area.children = rows_ui

    def _collect_panels(self):
        panels = []
        for pw in self.panel_widgets:
            panels.append(dict(caption=pw["caption"].value,
                               elev=pw["elev"].value, azim=pw["azim"].value,
                               zoom=pw["zoom"].value, legend=pw["legend"].value,
                               celldim=pw["celldim"].value,
                               compass=pw["compass"].value))
        return panels

    def _current_page(self):
        return (float(self.w_pw.value), float(self.w_ph.value))

    def _on_preview_pdf(self, _):
        if self.m.atoms is None:
            return
        fig = self.m.make_pdf_figure(
            self._current_page(), self.w_rows.value, self.w_cols.value,
            self.w_margin.value, self.w_wspace.value, self.w_hspace.value,
            self._collect_panels(), self.w_title.value)
        with self.out:
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
        self._status("PDF layout preview shown above.")

    def _on_export_pdf(self, _):
        if self.m.atoms is None:
            return
        fig = self.m.make_pdf_figure(
            self._current_page(), self.w_rows.value, self.w_cols.value,
            self.w_margin.value, self.w_wspace.value, self.w_hspace.value,
            self._collect_panels(), self.w_title.value)
        path = "/content/structure.pdf" if os.path.isdir("/content") else "structure.pdf"
        with PdfPages(path) as pdf:
            pdf.savefig(fig, facecolor=fig.get_facecolor())
        plt.close(fig)
        self._status(f"Exported {path}")
        try:
            from google.colab import files
            files.download(path)
        except Exception:
            self._status(f"Exported {path} (download only works in Colab).")

    # =======================================================================
    # Sync helpers between widgets and model
    # =======================================================================
    def _pull_settings(self):
        m = self.m
        m.bond_cutoff = self.w_bond.value
        m.show_legend = self.w_legend.value
        m.show_cell = self.w_cell.value
        m.show_axes = self.w_axes.value
        m.show_cell_dim = self.w_celldim.value
        m.perspective = self.w_persp.value
        m.depthshade = self.w_shade.value
        m.proportional_axes = self.w_propax.value
        m.hide_degenerate_axes = self.w_hidedeg.value
        m.cell_dim_style = self.w_dimstyle.value
        m.cell_dim_corner = self.w_dimcorner.value
        m.cell_dim_corner_dx = self.w_dim_dx.value
        m.cell_dim_corner_dy = self.w_dim_dy.value
        m.show_compass = self.w_compass.value
        m.compass_labels = self.w_compass_lbl.value
        m.compass_corner = self.w_compass_corner.value
        m.compass_dx = self.w_compass_dx.value
        m.compass_dy = self.w_compass_dy.value
        m.legend_corner = self.w_legend_corner.value
        m.legend_dx = self.w_legend_dx.value
        m.legend_dy = self.w_legend_dy.value
        m.atom_size_scale = self.w_size.value
        m.box_zoom = self.w_boxzoom.value
        m.elev = self.w_elev.value
        m.azim = self.w_azim.value
        m.zoom = self.w_zoom.value

    def _sync_cell_widgets(self):
        if self.m.atoms is None:
            return
        a, b, c, al, be, ga = cell_to_params(self.m.atoms.cell)
        self._suspend = True
        self.w_a.value, self.w_b.value, self.w_c.value = a, b, c
        self.w_al.value, self.w_be.value, self.w_ga.value = al, be, ga
        self._suspend = False

    def _clear_append_widgets(self):
        self._suspend = True
        for w in (self.w_ax_lo, self.w_ax_hi, self.w_by_lo, self.w_by_hi,
                  self.w_cz_lo, self.w_cz_hi):
            w.value = 0.0
        self._suspend = False

    def _rebuild_colors(self):
        self.color_pickers = {}
        rows = []
        if self.m.atoms is not None:
            syms = self.m.atoms.get_chemical_symbols()
            counts = {}
            for s in syms:
                counts[s] = counts.get(s, 0) + 1
            for el in sorted(counts, key=lambda s: atomic_numbers.get(s, 0)):
                cp = widgets.ColorPicker(
                    value=rgb_to_hex(self.m.atom_colors.get(el, (.5,.5,.5))),
                    description=f"{el} ({counts[el]})",
                    layout=widgets.Layout(width="220px"))
                cp.observe(self._make_color_cb(el), names="value")
                self.color_pickers[el] = cp
                rows.append(cp)
        self.color_box.children = rows

    def _make_color_cb(self, el):
        def _cb(change):
            self.m.atom_colors[el] = hex_to_rgb(change["new"])
            self._render()
        return _cb

    def _rebuild_pair_dropdowns(self):
        if self.m.atoms is None:
            return
        elems = sorted(set(self.m.atoms.get_chemical_symbols()),
                       key=lambda s: atomic_numbers.get(s, 0))
        self.w_pair_e1.options = elems
        self.w_pair_e2.options = elems
        if len(elems) > 1:
            self.w_pair_e2.value = elems[1]
        self.w_del_elem.options = elems

    def _rebuild_pair_rows(self):
        rows = []
        for key, val in sorted(self.m.pair_cutoffs.items()):
            e1, e2 = key
            ft = widgets.BoundedFloatText(value=val, min=0.0, max=50.0, step=0.05,
                                          description=f"{e1}–{e2} Å",
                                          layout=widgets.Layout(width="180px"))
            ft.observe(self._make_pair_cb(key), names="value")
            rm = widgets.Button(description="✕", layout=widgets.Layout(width="40px"))
            rm.on_click(self._make_pair_remove_cb(key))
            rows.append(widgets.HBox([ft, rm]))
        self.pair_box.children = rows

    def _make_pair_cb(self, key):
        def _cb(change):
            self.m.pair_cutoffs[key] = change["new"]
            self._render()
        return _cb

    def _make_pair_remove_cb(self, key):
        def _cb(_):
            self.m.pair_cutoffs.pop(key, None)
            self._rebuild_pair_rows()
            self._render()
        return _cb

    def _refresh_atom_list(self):
        if self.m.atoms is None:
            self.w_atomlist.options = []
            self.atoms_info.value = ""
            return
        pos, syms = self.m.get_display_atoms()
        coord = self.m.compute_coordination(pos, syms)
        n_real = len(self.m.atoms)
        opts = []
        cap = 4000
        for i in range(min(len(syms), cap)):
            tag = "" if i < n_real else "  (appended)"
            b = int(coord[i]) if i < len(coord) else 0
            flag = "  ⚠0-bond" if b == 0 else ""
            label = (f"{i:>4}  {syms[i]:<3}  "
                     f"({pos[i,0]:.2f}, {pos[i,1]:.2f}, {pos[i,2]:.2f})  "
                     f"bonds={b}{flag}{tag}")
            opts.append((label, i))
        self.w_atomlist.options = opts
        # info summary
        formula = self.m.atoms.get_chemical_formula()
        a, b, c, al, be, ga = cell_to_params(self.m.atoms.cell)
        vol = self.m.atoms.get_volume()
        self.atoms_info.value = (
            f"<b>Formula:</b> {formula} &nbsp; <b>Atoms:</b> {len(self.m.atoms)} "
            f"(displayed {len(syms)})<br>"
            f"<b>a,b,c:</b> {a:.3f}, {b:.3f}, {c:.3f} Å &nbsp; "
            f"<b>α,β,γ:</b> {al:.2f}, {be:.2f}, {ga:.2f}° &nbsp; "
            f"<b>V:</b> {vol:.2f} Å³")

    # ---- public display ---------------------------------------------------
    def show(self):
        display(self.ui)
        self._render()
        return self


# ===========================================================================
# Entry point
# ===========================================================================
def launch():
    """Build, display, and return the visualizer app."""
    app = CrystalVisualizerColab()
    return app.show()


if __name__ == "__main__":
    # When run as a script (not in a notebook) just do a smoke test.
    print("This module is intended to be used inside a Colab/Jupyter notebook.")
    print("In a notebook cell run:\n    from crystal_visualizer_colab import launch"
          "\n    app = launch()")

app = launch()

This module is intended to be used inside a Colab/Jupyter notebook.
In a notebook cell run:
    from crystal_visualizer_colab import launch
    app = launch()
